# 🔧 Day 01a: sklearn Preprocessing — fit/transform Mastery

---

## 🎯 What You'll Master Today
- StandardScaler, MinMaxScaler
- LabelEncoder, OneHotEncoder, OrdinalEncoder
- SimpleImputer, KNNImputer
- fit/transform pattern

**Goal:** Write a preprocessing pipeline from memory.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  PREPROCESSING — fit/transform PATTERN                           ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Training:   scaler.fit_transform(X_train)                      ║
║  Testing:    scaler.transform(X_test)     ← NO fit!             ║
║                                                                   ║
║  ⚠️ NEVER fit on test data → data leakage!                     ║
║                                                                   ║
║  NUMERIC:                                                        ║
║    StandardScaler  → (x - μ) / σ     (mean=0, std=1)           ║
║    MinMaxScaler    → (x - min) / (max - min)  [0, 1]           ║
║                                                                   ║
║  CATEGORICAL:                                                    ║
║    OrdinalEncoder  → integers (when order matters)              ║
║    OneHotEncoder   → binary columns (no order)                  ║
║    LabelEncoder    → target variable encoding                   ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    LabelEncoder, OneHotEncoder, OrdinalEncoder
)
from sklearn.impute import SimpleImputer, KNNImputer

# ============================================================
# 1. Scaling — StandardScaler vs MinMaxScaler
# ============================================================

X = np.array([[1, 100], [2, 200], [3, 300], [4, 400], [5, 500]])

# StandardScaler: (x - mean) / std → mean=0, std=1
ss = StandardScaler()
X_standard = ss.fit_transform(X)
print(f"StandardScaler:")
print(f"  Mean: {X_standard.mean(axis=0).round(10)}")
print(f"  Std:  {X_standard.std(axis=0).round(10)}")
print(f"  Data: {X_standard[:3].round(3)}\n")

# MinMaxScaler: (x - min) / (max - min) → [0, 1]
mm = MinMaxScaler()
X_minmax = mm.fit_transform(X)
print(f"MinMaxScaler:")
print(f"  Range: [{X_minmax.min()}, {X_minmax.max()}]")
print(f"  Data: {X_minmax[:3].round(3)}\n")

# RobustScaler: (x - median) / IQR → robust to outliers
rs = RobustScaler()
X_robust = rs.fit_transform(X)
print(f"RobustScaler: {X_robust[:3].round(3)}")

print("\n💡 StandardScaler = default. RobustScaler when outliers present.")

In [ ]:
# ============================================================
# 2. Encoding Categorical Variables
# ============================================================

# LabelEncoder — for target variable
le = LabelEncoder()
labels = ['cat', 'dog', 'bird', 'cat', 'dog']
encoded = le.fit_transform(labels)
print(f"LabelEncoder: {labels} → {encoded}")
print(f"Classes: {le.classes_}")
print(f"Inverse: {le.inverse_transform([0, 1, 2])}\n")

# OrdinalEncoder — when order matters
oe = OrdinalEncoder(categories=[['low', 'medium', 'high']])
sizes = np.array([['medium'], ['high'], ['low'], ['high']])
encoded = oe.fit_transform(sizes)
print(f"OrdinalEncoder: {sizes.flatten()} → {encoded.flatten()}\n")

# OneHotEncoder — no order (nominal categories)
ohe = OneHotEncoder(sparse_output=False, drop='first')  # drop='first' avoids multicollinearity
colors = np.array([['red'], ['blue'], ['green'], ['red']])
encoded = ohe.fit_transform(colors)
print(f"OneHotEncoder (drop first):")
print(f"  Categories: {ohe.categories_[0]}")
print(f"  Encoded:\n{encoded}")

print("\n💡 OneHot for features (no order). Label/Ordinal for ordered or target.")

In [ ]:
# ============================================================
# 3. Imputing Missing Values
# ============================================================

X_missing = np.array([[1, 2], [np.nan, 3], [7, np.nan], [4, 5], [np.nan, np.nan]])
print(f"Data with NaN:\n{X_missing}\n")

# SimpleImputer — mean (default)
imp_mean = SimpleImputer(strategy='mean')
X_filled = imp_mean.fit_transform(X_missing)
print(f"Mean imputed:\n{X_filled.round(2)}\n")

# SimpleImputer — median (better with outliers)
imp_median = SimpleImputer(strategy='median')
X_filled2 = imp_median.fit_transform(X_missing)
print(f"Median imputed:\n{X_filled2.round(2)}\n")

# SimpleImputer — most_frequent (categorical)
cats = np.array([['red'], ['blue'], [None], ['red'], [None]])
imp_freq = SimpleImputer(strategy='most_frequent')
print(f"Most frequent: {imp_freq.fit_transform(cats).flatten()}\n")

# KNNImputer — uses K nearest neighbors to impute
imp_knn = KNNImputer(n_neighbors=2)
X_knn = imp_knn.fit_transform(X_missing)
print(f"KNN imputed:\n{X_knn.round(2)}")

print("\n💡 KNNImputer is smarter but slower. SimpleImputer is usually enough.")

In [ ]:
# ============================================================
# 4. ⚠️ The Data Leakage Trap
# ============================================================

from sklearn.model_selection import train_test_split

X = np.random.randn(100, 3) * 10 + 50
y = (X[:, 0] > 50).astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ❌ WRONG — fitting scaler on ALL data
# scaler = StandardScaler()
# X_all_scaled = scaler.fit_transform(X)  ← leaks test info into training!
# X_train_scaled, X_test_scaled = X_all_scaled[:80], X_all_scaled[80:]

# ✅ CORRECT — fit on train, transform test
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # fit + transform on train
X_test_s = scaler.transform(X_test)          # only transform on test

print(f"Train mean: {X_train_s.mean(axis=0).round(10)}")
print(f"Test mean:  {X_test_s.mean(axis=0).round(4)}")  # NOT exactly 0 — correct!

print("\n⚠️ RULE: fit_transform(train), transform(test). NEVER fit on test!")

---

## 🗺️ Preprocessing Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  PREPROCESSING — Decision Guide                                  ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  SCALING:                                                        ║
║    StandardScaler → default (SVM, KNN, regression)              ║
║    MinMaxScaler   → when you need [0,1] range                   ║
║    RobustScaler   → when outliers present                       ║
║    None needed    → trees, random forest, XGBoost               ║
║                                                                   ║
║  ENCODING:                                                       ║
║    OneHotEncoder  → nominal categories (color, city)            ║
║    OrdinalEncoder → ordered categories (low/med/high)           ║
║    LabelEncoder   → target variable only                        ║
║                                                                   ║
║  MISSING:                                                        ║
║    SimpleImputer(mean)    → numeric, no outliers                ║
║    SimpleImputer(median)  → numeric, with outliers              ║
║    SimpleImputer(frequent)→ categorical                         ║
║    KNNImputer             → smarter, considers neighbors        ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | fit_transform on train vs test? | fit_transform on train, transform-only on test. Prevents data leakage |
| 2 | StandardScaler vs MinMaxScaler? | Standard: mean=0, std=1 (for SVM, KNN). MinMax: [0,1] range (for neural nets) |
| 3 | When is scaling NOT needed? | Tree-based models (DT, RF, XGBoost) — they split on thresholds, not distances |
| 4 | OneHot vs Label encoding? | OneHot for nominal (no order). Label for ordinal or target. OneHot avoids false ordering |
| 5 | What is data leakage? | Test data info leaking into training (e.g., scaling on full data). Leads to overly optimistic results |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • fit_transform(train) → transform(test)              │
## │  • StandardScaler = default for most models             │
## │  • OneHot for nominal, Ordinal for ordered              │
## │  • SimpleImputer(median) for robustness                 │
## │  • Trees don't need scaling                             │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Feature Engineering** — polynomial, interaction, selection